In [1]:
import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler
import timm

In [10]:
!gdown 1B8W2fglUmUAg-9At2nKTcMiGRMW9qQLM

Downloading...
From (original): https://drive.google.com/uc?id=1B8W2fglUmUAg-9At2nKTcMiGRMW9qQLM
From (redirected): https://drive.google.com/uc?id=1B8W2fglUmUAg-9At2nKTcMiGRMW9qQLM&confirm=t&uuid=e6c16ac7-8be7-477e-9706-7c4ae755e762
To: /content/super-ai-engineer-season-6-individual-hackathon-house-recognition.zip
100% 1.34G/1.34G [00:09<00:00, 141MB/s]


In [22]:
!unzip /content/super-ai-engineer-season-6-individual-hackathon-house-recognition.zip

Archive:  /content/super-ai-engineer-season-6-individual-hackathon-house-recognition.zip
  inflating: sample_submission.csv   
  inflating: test/test/00162f19.jpg  
  inflating: test/test/004c4789.jpg  
  inflating: test/test/0059b42f.jpg  
  inflating: test/test/005f930c.jpg  
  inflating: test/test/009095e8.jpg  
  inflating: test/test/00a98706.jpg  
  inflating: test/test/00da9dd7.jpg  
  inflating: test/test/00e2bfe0.jpg  
  inflating: test/test/00eac2a3.jpg  
  inflating: test/test/01265567.jpg  
  inflating: test/test/0188f57d.jpg  
  inflating: test/test/01945b53.jpg  
  inflating: test/test/01ccbb76.jpg  
  inflating: test/test/01e45f69.jpg  
  inflating: test/test/01ee5917.jpg  
  inflating: test/test/01f9d203.jpg  
  inflating: test/test/0264a981.jpg  
  inflating: test/test/0272a96c.jpg  
  inflating: test/test/0282e334.jpg  
  inflating: test/test/02abb751.jpg  
  inflating: test/test/0338f3c6.jpg  
  inflating: test/test/034c3c52.jpg  
  inflating: test/test/0356ad78.jpg  

In [ ]:
TRAIN_CSV = '/content/train.csv' 
TRAIN_IMG_DIR = '/content/train/train/'
TEST_IMG_DIR = '/content/test/test'

BATCH_SIZE = 2
NUM_EPOCHS_WARMUP = 2
NUM_EPOCHS_FULL = 5
LEARNING_RATE_HEAD = 1e-4
LEARNING_RATE_FULL = 1e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = 'eva02_large_patch14_448.mim_m38m_ft_in22k_in1k'

In [3]:
train_csv = pd.read_csv(TRAIN_CSV)

In [4]:
train_csv['class'].unique()

array([0, 1])

In [5]:
class HouseTrainDataset(Dataset):
    """ สำหรับโหลดข้อมูล Train จาก CSV """
    def __init__(self, csv_file, img_dir, transform=None):
        self.data_info = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_info)

    def __getitem__(self, idx):
        # คอลัมน์ 0 = ชื่อไฟล์, คอลัมน์ 1 = Label
        img_name = str(self.data_info.iloc[idx, 0])
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        label = int(self.data_info.iloc[idx, 1])

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
class HouseInferenceDataset(Dataset):
    """ สำหรับโหลดข้อมูล Test (ไม่มี Label) เพื่อทำ Prediction """
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        # ดึงรายชื่อไฟล์ทั้งหมดและเรียงลำดับ
        self.image_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, img_name

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((448, 448)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((448, 448)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

print("-> Preparing DataLoaders...")
train_dataset = HouseTrainDataset(csv_file=TRAIN_CSV, img_dir=TRAIN_IMG_DIR, transform=data_transforms['train'])
test_dataset = HouseInferenceDataset(img_dir=TEST_IMG_DIR, transform=data_transforms['test'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)

-> Preparing DataLoaders...


In [ ]:
print(f"-> Loading Model: {MODEL_NAME}...")
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=2)

model.set_grad_checkpointing(True)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

ACCUMULATION_STEPS = 8

def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    optimizer.zero_grad()

    for i, (inputs, labels) in enumerate(loader):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss = loss / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        # อัปเดตน้ำหนักเฉพาะตอนสะสมครบ 8 รอบ หรือหมด Dataset แล้ว
        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * inputs.size(0) * ACCUMULATION_STEPS
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100. * correct / total

-> Loading Model: eva02_large_patch14_448.mim_m38m_ft_in22k_in1k...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/tmp/ipykernel_29134/223789614.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # กลับมาใช้เพื่อช่วยลด VRAM


In [ ]:
print("\n[Phase 1] Training only the classification head...")

for param in model.parameters():
    param.requires_grad = False
    
for param in model.head.parameters():
    param.requires_grad = True

optimizer_head = optim.AdamW(model.head.parameters(), lr=LEARNING_RATE_HEAD)

for epoch in range(NUM_EPOCHS_WARMUP):
    loss, acc = train_one_epoch(model, train_loader, optimizer_head)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS_WARMUP} | Loss: {loss:.4f} | Acc: {acc:.2f}%")


[Phase 1] Training only the classification head...


/tmp/ipykernel_29134/223789614.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/2 | Loss: 0.2958 | Acc: 91.20%
Epoch 2/2 | Loss: 0.1672 | Acc: 94.58%


In [ ]:
print("\n[Phase 2] Fine-tuning all layers...")
for param in model.parameters():
    param.requires_grad = True

optimizer_full = optim.AdamW(model.parameters(), lr=LEARNING_RATE_FULL)

for epoch in range(NUM_EPOCHS_FULL):
    loss, acc = train_one_epoch(model, train_loader, optimizer_full)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS_FULL} | Loss: {loss:.4f} | Acc: {acc:.2f}%")

torch.save(model.state_dict(), f'{MODEL_NAME}_house_classifier.pth')
print(f"-> Model saved to '{MODEL_NAME}_house_classifier.pth'")


[Phase 2] Fine-tuning all layers...


/tmp/ipykernel_29134/223789614.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/5 | Loss: 0.1094 | Acc: 95.97%
Epoch 2/5 | Loss: 0.0322 | Acc: 99.15%
Epoch 3/5 | Loss: 0.0062 | Acc: 99.90%
Epoch 4/5 | Loss: 0.0006 | Acc: 100.00%
Epoch 5/5 | Loss: 0.0003 | Acc: 100.00%
-> Model saved to 'eva02_large_patch14_448.mim_m38m_ft_in22k_in1k_house_classifier.pth'


In [ ]:
print("\n[Phase 3] Starting Prediction on Test Set...")
model.eval()
results = []

with torch.no_grad():
    for inputs, filenames in test_loader:
        inputs = inputs.to(DEVICE)

        with autocast():
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

        for i in range(len(filenames)):

            current_name = filenames[i]
            clean_name = current_name.replace(".jpg", "")

            results.append({
                'id': clean_name,
                'answer': predicted[i].item()
            })

df_submission = pd.DataFrame(results)
df_submission.to_csv('submission.csv', index=False)
print("-> Finished! Predictions saved to 'submission.csv'")

print("\nPreview of submission.csv:")
print(df_submission.head())


[Phase 3] Starting Prediction on Test Set...


/tmp/ipykernel_29134/753274369.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


-> Finished! Predictions saved to 'submission.csv'

Preview of submission.csv:
         id  answer
0  00162f19       0
1  004c4789       0
2  0059b42f       1
3  005f930c       0
4  009095e8       1
